In [ ]:
import pandas as pd
import re

In [ ]:
FILE_PATH = "2024_All_CAP_Search_Results_Data_P14.xlsx"  # adjust path as needed
 
# sheet_name=None reads ALL sheets into a dict of {sheet_name: DataFrame}
all_sheets = pd.read_excel(FILE_PATH, sheet_name=None)
 
print("Sheets found:", list(all_sheets.keys()))
# Expect something like: ['RPA', 'WG', 'SGRPID', 'DAERA'] — one per UK Paying Agency.
# You only need RPA (England) for YNYCA, but combining all is useful to sanity-check
# your filter isn't accidentally catching non-English postcodes.
 
# Use .assign() + concat in one step to avoid the "highly fragmented DataFrame"
# performance warning you'd get from repeatedly inserting a column in a loop.
combined = pd.concat(
    [df.assign(source_agency=name) for name, df in all_sheets.items()],
    ignore_index=True,
)
print(f"Total rows across all agencies: {len(combined):,}")
print("Columns:", combined.columns.tolist())
 
# Real sheet names observed: ['DARDNI', 'RPA_1', 'RPA2', 'SGRPID', 'WG']
# RPA (England) is split across TWO tabs — RPA_1 and RPA2 — probably because
# the full England dataset exceeds a single sheet's row limit. Both are needed
# for complete England coverage; DARDNI/SGRPID/WG are NI/Scotland/Wales.
# Filtering to England only up front is optional (postcode filtering below
# will naturally exclude the other nations), but it's a bit faster:
england_only = combined[combined["source_agency"].isin(["RPA_1", "RPA2"])].copy()
print(f"England (RPA_1 + RPA2) rows: {len(england_only):,}")

In [ ]:
# Join against  Join against ONS postcode-to-CAUTH lookup
# Download the lookup first from:
# https://open-geography-portalx-ons.hub.arcgis.com/datasets/ons::postcode-to-oa-2021-to-lsoa-to-msoa-to-ltla-to-utla-to-cauth-november-2023-best-fit-lookup-in-the-uk

In [ ]:
POSTCODE_COL = "PostcodePrefix_F202B"
NSPL_PATH = "PCD_OA21_LSOA21_MSOA21_LTLA22_UTLA22_CAUTH22_NOV23_UK_LU_v2.csv"  # adjust to actual downloaded filename
 
def precise_filter(combined_df, nspl_path, postcode_col=POSTCODE_COL):
    """
    Join CAP payments data (outward postcode codes only, e.g. "YO18") against
    the ONS NSPL file to identify YNYCA-area rows.
 
    IMPORTANT DISCOVERY: this NSPL vintage predates YNYCA's formal recognition
    as a Combined Authority (the mayor wasn't elected until May 2024), so the
    'cauth22nm' field is likely BLANK for York/North Yorkshire rows even though
    the geography obviously belongs there. Don't rely on cauth22nm alone.
 
    Also: 'ltla22nm' reflects the OLD pre-2023 North Yorkshire district names
    (Craven, Hambleton, Harrogate, Richmondshire, Ryedale, Scarborough, Selby),
    NOT "North Yorkshire" itself. "North Yorkshire" only appears at the
    'utla22nm' (upper-tier) level. So we filter on utla22nm, which is reliable
    regardless of the CAUTH geography's currency.
    """
    nspl = pd.read_csv(
        nspl_path,
        low_memory=False,
        usecols=["pcds", "ltla22nm", "utla22nm", "cauth22nm"],
    )
 
    def outward_from_full(pc):
        if pd.isna(pc):
            return None
        return str(pc).strip().upper().split(" ")[0]
 
    nspl["outward_code"] = nspl["pcds"].apply(outward_from_full)
 
    # Collapse to one row per outward code (mode handles rare edge-of-boundary cases)
    lookup = (
        nspl.groupby("outward_code")
        .agg(
            utla22nm=("utla22nm", lambda s: s.mode().iat[0] if not s.mode().empty else None),
            ltla22nm=("ltla22nm", lambda s: s.mode().iat[0] if not s.mode().empty else None),
            cauth22nm=("cauth22nm", lambda s: s.mode().iat[0] if not s.mode().empty else None),
        )
        .reset_index()
    )
    print(f"Built lookup for {len(lookup):,} unique outward codes")
 
    # Diagnostic: check whether CAUTH is actually populated for YNYCA at all
    ynyca_check = lookup[lookup["utla22nm"].isin(["York", "North Yorkshire"])]
    cauth_populated = ynyca_check["cauth22nm"].notna().sum()
    print(f"YNYCA-area outward codes with a non-blank CAUTH value: "
          f"{cauth_populated} / {len(ynyca_check)}")
    if cauth_populated == 0:
        print("  -> CONFIRMED: cauth22nm is blank for YNYCA in this file vintage. "
              "Using utla22nm as the filter field instead.")
 
    def clean_pc(pc):
        if pd.isna(pc):
            return None
        return str(pc).strip().upper().replace(" ", "")
 
    combined_df = combined_df.copy()
    combined_df["_outward_clean"] = combined_df[postcode_col].apply(clean_pc)
 
    merged = combined_df.merge(
        lookup,
        left_on="_outward_clean",
        right_on="outward_code",
        how="left",
    )
 
    # Use utla22nm (reliable) rather than cauth22nm (likely blank for YNYCA)
    ynyca_precise = merged[merged["utla22nm"].isin(["York", "North Yorkshire"])].copy()
 
    return ynyca_precise
 
 
# Run it (uncomment once you've downloaded the NSPL file locally):
precise_result = precise_filter(combined, NSPL_PATH)
print(f"Precise YNYCA match: {len(precise_result):,} rows")

 
 

In [ ]:
precise_result.to_csv("ynyca_cap_payments_precise.csv", index=False)

In [ ]:
#Tidy (long-format) totals by subsidy scheme, YNYCA, 2024
# ---------------------------------------------------------------------------
# Goal: not farm-level detail — just "how much did YNYCA get, per scheme,
# in 2024?" Melting from wide to long format makes this a one-line groupby.
 
# Use whichever filtered result you trust most — swap in `precise_result`
# here once you've run precise_filter() with the NSPL file.
ynyca_data = precise_result
 
# Restrict to 2024 (adjust column name if 'Year' differs, e.g. 'CAL_YR')
ynyca_2024 = ynyca_data[ynyca_data["Year"] == 2024].copy()
print(f"\nYNYCA rows for 2024: {len(ynyca_2024):,}")
 
# Identify which columns are "identifying/summary" columns vs. actual
# per-scheme measure columns — everything NOT in this list is a measure column.
ID_AND_SUMMARY_COLS = [
    "Year", "BeneficiaryCode", "BeneficiaryName_F201", "PostcodePrefix_F202B",
    "TownCity_F202C", "OtherEAGFTotal", "DirectEAGFTotal", "RuralDevelopmentTotal",
    "Total", "PayingAgencyLink", "source_agency", "outward_code",
]
measure_cols = [c for c in ynyca_2024.columns if c not in ID_AND_SUMMARY_COLS]
print(f"Number of individual scheme/measure columns: {len(measure_cols)}")
 
# Melt: one row per (beneficiary, scheme, amount) instead of one row per
# beneficiary with 115 mostly-empty columns.
tidy = ynyca_2024.melt(
    id_vars=["BeneficiaryCode", "PostcodePrefix_F202B", "Year"],
    value_vars=measure_cols,
    var_name="scheme",
    value_name="amount",
)
 
# Drop the zero/blank cells — most beneficiary-scheme combinations are empty,
# since a given farm typically only claims a handful of the ~115 schemes.
tidy["amount"] = pd.to_numeric(tidy["amount"], errors="coerce")
tidy = tidy[tidy["amount"].notna() & (tidy["amount"] != 0)]
print(f"Tidy (non-zero) rows: {len(tidy):,}")
 
# how much funding did YNYCA farms get, by scheme, in 2024:
scheme_totals_2024 = (
    tidy.groupby("scheme")["amount"]
    .agg(total_paid="sum", num_recipients="count")
    .sort_values("total_paid", ascending=False)
)
 
print("\nYNYCA subsidy totals by scheme, 2024:")
print(scheme_totals_2024.to_string())
 
scheme_totals_2024.to_csv("ynyca_scheme_totals_2024.csv")
print("\nSaved -> ynyca_scheme_totals_2024.csv")
 
# Also worth checking the three big-picture totals directly (no melt needed):
print(f"\nYNYCA overall 2024 totals:")
print(f"  Direct Aid (BPS/delinked):  £{ynyca_2024['DirectEAGFTotal'].sum():,.2f}")
print(f"  Rural Development:          £{ynyca_2024['RuralDevelopmentTotal'].sum():,.2f}")
print(f"  Other (Market Schemes):     £{ynyca_2024['OtherEAGFTotal'].sum():,.2f}")
print(f"  TOTAL:                       £{ynyca_2024['Total'].sum():,.2f}")


print(f"Note that Sustaintainable Financial Investment funds are not included in these totals.")


YNYCA rows for 2024: 5,589
Number of individual scheme/measure columns: 113
Tidy (non-zero) rows: 6,369

YNYCA subsidy totals by scheme, 2024:
                                                                  total_paid  num_recipients
scheme                                                                                      
Basic payment scheme - UK funded                                 35843861.89            5479
Agri-environment-climate - UK funded                              9358736.38             514
Young farmers - UK funded                                          339854.50             147
Investment in forest area development and viability - UK funded    187084.08             130
Investments in physical assets - UK funded                         136817.62              10
Greening - UK payments - UK funded                                  64230.22              37
Forest environmental, climate and conservation - UK funded          15993.47               4
Investment in fores